In [ ]:
import json
import random
from bs4 import BeautifulSoup #html analyis
from selenium import webdriver #enable javascript on the website
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from pyvirtualdisplay import Display
from urllib.parse import urlparse #Extract domain from URL

In [ ]:
from str_compare import similar
from os.path import exists

def mkfilt(domain,urls_file):
    urls=[]
    if not exists(urls_file):
        return False
    with open(urls_file, mode="r") as f:        
        url = f.readline()
        while(url != ""):
            if domain in url:
                urls.append(url[:-1])
            url = f.readline()
    if len(urls)<2:
        return False
    r=random.randint(0,len(urls)-1)
    url_a=urls[r]
    urls.pop(r)
    r=random.randint(0,len(urls)-1)
    url_b=urls[r]
    a=InfoFinder(url_a)
    a_content = a.get_content(filt=False)
    #print(a.topic.toJson())
    b=InfoFinder(url_b)
    b_content = b.get_content(filt=False)
    #print(b.topic.toJson())

    skip=[]        
    for i in a_content:
        for j in b_content:
            if similar(i,j,0.6):
                skip.append(i)
                break
    print(skip)
    with open("./files/content_skip/" + a.get_domain() + ".filt", mode="w") as f:
        for i in skip:
            f.write(i.replace("\n",""))
            f.write("\n")
    return True

In [ ]:
class Topic:
    def __init__(self,url="",title="",content="",time=""):
        self.url=url
        self.title=title
        self.content=content
        self.time=time
    def toJson(self):
        return json.dumps(self,
                          default=lambda o: o.__dict__,
                          sort_keys=True,
                          indent=4
                          ,ensure_ascii=False)    

In [ ]:
import time
class InfoFinder:
    __school=""
    topic=""
    __skip=["http","報名"]
    __html=""
    __domain=""
    def __init__(self,url,school=""):
        self.topic=Topic(url=url)
        self.school=school
        self.get_html()   
        
    def get_html(self):
        display = Display(visible=0, size=(1920, 1080))  
        display.start()
        chrome_options = Options()
        #chrome_options.add_argument('--headless')
        chrome_options.add_argument('ignore-certificate-errors')
        chrome_options.add_argument('disable-gpu')
        chrome_options.add_argument('--incognito')
        ser = Service(ChromeDriverManager().install())
        desired_capabilities = DesiredCapabilities.CHROME.copy() 
        desired_capabilities['acceptInsecureCerts'] = True
        browser = webdriver.Chrome(service=ser,options=chrome_options, desired_capabilities=desired_capabilities)
        browser.get(self.topic.url)
        time.sleep(3)
        self.html = browser.page_source
        self.topic.url = browser.current_url
        self.soup = BeautifulSoup(browser.page_source, "html.parser")
        browser.quit()
        display.stop()

    def get_title(self):
        #print(self.topic.url)
        #print(self.html)
        title = self.soup.find("title")
        self.topic.title = title.string
        return self.topic.title

    def get_content(self,filt=True):
        content_html = self.soup.find("body")
        text = content_html.get_text()
        texts = []
        for i in text.split("\n\n\n"):
            if i != "":
                texts.append(i.replace("     ", ""))                
        self.topic.content=texts
        if filt:
            self.__filt_content()
        return self.topic.content
    
    def get_domain(self):
        self.domain = urlparse(self.topic.url).netloc
        return self.domain
    
    def __filt_content(self):
        domain = urlparse(self.topic.url).netloc        
        try:
            f = open("./files/content_skip/" + a.get_domain() + ".filt",mode='r')            
        except:
            print('making: ' + "./files/content_skip/" + a.get_domain() + ".filt")
            if mkfilt(a.get_domain(),"./files/urls/"+self.school+".txt"):
                f = open("./files/content_skip/" + a.get_domain() + ".filt",mode='r')
            else:
                return False
        
        filt_texts=[]
        filt_text = f.readline()
        while(filt_text != ""):
            filt_texts.append(filt_text[:-1])
            filt_text = f.readline()
        f.close() 
        contents = []
        for i in self.topic.content:
            for j in filt_texts:
                if similar(i,j,0.7):
                    break
            else:
                contents.append(i)
        self.topic.content = contents
        return True